In [ ]:
import os
import urllib.request
import subprocess

BASE_DRIVE = "/content/drive/MyDrive/Distil_Whisper_Vi_Yanshee"
HF_MODEL_DIR = f"{BASE_DRIVE}/models/distil_phowhisper_base_trained"
GGML_OUT_DIR = f"{BASE_DRIVE}/models/whisper_cpp_ggml"
os.makedirs(GGML_OUT_DIR, exist_ok=True)

# 1. TẢI TRỰC TIẾP 2 FILE vocab.json VÀ added_tokens.json TỪ HUGGINGFACE HUB
print("📥 [1/3] Đang tải trực tiếp vocab.json và added_tokens.json vào thư mục model...")
for fname in ["vocab.json", "added_tokens.json", "merges.txt"]:
    url = f"https://huggingface.co/openai/whisper-base/resolve/main/{fname}"
    dest = os.path.join(HF_MODEL_DIR, fname)
    urllib.request.urlretrieve(url, dest)
    print(f"   ✅ Đã có file: {dest} ({os.path.getsize(dest)//1024} KB)")

# Đảm bảo đã clone whisper.cpp và whisper_openai nếu mới reset Colab
if not os.path.exists("/content/whisper.cpp"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggerganov/whisper.cpp.git", "/content/whisper.cpp"], check=True)
if not os.path.exists("/content/whisper_openai"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/openai/whisper.git", "/content/whisper_openai"], check=True)

# 2. CHẠY CHUYỂN ĐỔI SANG ĐỊNH DẠNG GGML (fp16)
print("\n🔄 [2/3] Đang chuyển đổi Distil-PhoWhisper sang định dạng ggml (fp16)...")
res = subprocess.run(
    ["python3", "/content/whisper.cpp/models/convert-h5-to-ggml.py", HF_MODEL_DIR, "/content/whisper_openai", GGML_OUT_DIR],
    capture_output=True, text=True
)
if res.returncode != 0:
    print("❌ Chi tiết lỗi:", res.stderr)
    raise RuntimeError("Convert thất bại")
print("✅ Đã tạo thành công ggml-model.bin (fp16)!")

# 3. BIÊN DỊCH WHISPER-QUANTIZE VÀ NÉN XUỐNG Q8_0 (INT8 CHO PI 3B armv7l)
print("\n🗜️ [3/3] Đang nén sang Q8_0 (INT8 - 8 bit) cho Raspberry Pi 3B (armv7l)...")
subprocess.run(["cmake", "-B", "/content/whisper.cpp/build", "/content/whisper.cpp"], check=True)
subprocess.run(["cmake", "--build", "/content/whisper.cpp/build", "--target", "whisper-quantize", "-j4"], check=True)

fp16_bin = os.path.join(GGML_OUT_DIR, "ggml-model.bin")
q8_bin = os.path.join(BASE_DRIVE, "ggml-distil-phowhisper-q8_0.bin")

subprocess.run(["/content/whisper.cpp/build/bin/whisper-quantize", fp16_bin, q8_bin, "q8_0"], check=True)

size_mb = os.path.getsize(q8_bin) / (1024 ** 2)
print("\n" + "=" * 75)
print(f"🎉 ĐÃ TẠO XONG FILE WHISPER.CPP INT8 (Q8_0) CHO RASPBERRY PI 3B (armv7l)!")
print(f"📦 Đường dẫn trên Google Drive: {q8_bin}")
print(f"💾 Dung lượng: {size_mb:.1f} MB (Chỉ 1 file duy nhất, chạy thẳng bằng C++ trên Pi 3B!)")
print("=" * 75)

📥 [1/3] Đang tải trực tiếp vocab.json và added_tokens.json vào thư mục model...
   ✅ Đã có file: /content/drive/MyDrive/Distil_Whisper_Vi_Yanshee/models/distil_phowhisper_base_trained/vocab.json (815 KB)
   ✅ Đã có file: /content/drive/MyDrive/Distil_Whisper_Vi_Yanshee/models/distil_phowhisper_base_trained/added_tokens.json (33 KB)
   ✅ Đã có file: /content/drive/MyDrive/Distil_Whisper_Vi_Yanshee/models/distil_phowhisper_base_trained/merges.txt (482 KB)

🔄 [2/3] Đang chuyển đổi Distil-PhoWhisper sang định dạng ggml (fp16)...
✅ Đã tạo thành công ggml-model.bin (fp16)!

🗜️ [3/3] Đang nén sang Q8_0 (INT8 - 8 bit) cho Raspberry Pi 3B (armv7l)...

🎉 ĐÃ TẠO XONG FILE WHISPER.CPP INT8 (Q8_0) CHO RASPBERRY PI 3B (armv7l)!
📦 Đường dẫn trên Google Drive: /content/drive/MyDrive/Distil_Whisper_Vi_Yanshee/ggml-distil-phowhisper-q8_0.bin
💾 Dung lượng: 60.8 MB (Chỉ 1 file duy nhất, chạy thẳng bằng C++ trên Pi 3B!)
